# First Principles: Numerical Linear Algebra & Iterative Solvers

## 1. Phenomenon

In physical modeling, partial differential equations (PDEs), optimization, and AI, we frequently encounter massive linear systems $Ax = b$ and eigenvalue problems $Ax = \lambda x$.
Exact analytical solutions (or exact direct methods) are computationally prohibitive for large dimensions ($n \ge 10^5$)
and subject to finite-precision floating-point arithmetic errors.

## 2. Goal

Solve or approximate $Ax = b$ and key eigenvalues/eigenvectors efficiently, accurately, and stably using finite-precision arithmetic by exploiting matrix properties (e.g., sparsity, symmetry, positive definiteness, spectral properties).

## 3. Assumptions

* Computations use IEEE 754 Floating-Point (FP) arithmetic, introducing roundoff errors.
* The system matrix $A \in \mathbb{R}^{n \times n}$ is often large and sparse (number of non-zero entries $\operatorname{nnz}(A) \ll n^2$).
* For specific solvers (e.g., Conjugate Gradient), $A$ is Symmetric Positive Definite (SPD).
* Matrix-vector multiplications (SpMV) are computationally cheap compared to matrix factorizations.

## 4. Variables

* $x_k \in \mathbb{R}^n$: Approximate solution vector at iteration $k$.
* $e_k = x - x_k \in \mathbb{R}^n$: Exact solution error vector at iteration $k$, where $Ax = b$.
* $r_k = b - A x_k = A e_k \in \mathbb{R}^n$: Residual vector at iteration $k$.
* $p_k \in \mathbb{R}^n$: Search direction vector at iteration $k$.

## 5. Parameters

* $A \in \mathbb{R}^{n \times n}$: Non-singular coefficient matrix.
* $b \in \mathbb{R}^n$: Right-hand side load vector.
* $\epsilon_{\text{mach}}$: Machine precision (unit roundoff).
* $\kappa(A) = \|A\| \|A^{-1}\|$: Condition number of $A$.
* $\omega \in (0, 2)$: Relaxation parameter for SOR.

## 6. Units and Domains

* $A$: Domain-dependent physical dimension (e.g., $\text{s}^{-1}$, $\text{N}\cdot\text{m}^{-1}$, or dimensionless).
* $b$: Domain-dependent forcing term.
* $x$: Inherits units from $A^{-1}b$.
* Condition number $\kappa(A)$ and spectral radius $\rho(G)$ are strictly dimensionless.

## 7. Floating Point Arithmetic & Machine Precision

Real numbers are represented in floating-point format as $x = \pm m \times \beta^e$, where $m$ is the mantissa, $\beta$ is the base ($\beta=2$), and $e$ is the exponent.

Under the standard IEEE 754 floating-point model, elementary arithmetic operations $\odot \in \{+, -, \times, /\}$ satisfy:

$$
\operatorname{fl}(x \odot y) = (x \odot y)(1 + \delta), \quad |\delta| \le \epsilon_{\text{mach}}
$$

For IEEE double precision (FP64), machine precision is $\epsilon_{\text{mach}} = 2^{-53} \approx 1.11 \times 10^{-16}$.

## 8. Flop Complexity & Sparse Formats

A floating-point operation (flop) is an elementary operation ($+, -, \times, /$).

* **Dense Matrix-Vector Product (MatVec)**: $2n^2 - n \approx 2n^2$ flops.
* **Dense LU Factorization**: $\frac{2}{3}n^3 + \mathcal{O}(n^2)$ flops.
* **Sparse Matrix-Vector Product (SpMV)**: $2 \cdot \operatorname{nnz}(A)$ flops.

Sparse matrices are stored using specialized data structures:

* **COO (Coordinate)**: Arrays of row indices, column indices, and values.
* **CSR (Compressed Sparse Row)**: `values`, `column_indices`, and `row_pointers` arrays. Enables cache-efficient $\mathcal{O}(\operatorname{nnz})$ SpMV operations.
* **CSC (Compressed Sparse Column)**: Column-oriented counterpart of CSR, optimized for column access.

## 9. Forward vs Backward Numerical Stability

Let $y = f(x)$ be a mathematical function and $\hat{y} = \operatorname{alg}(x)$ be the computed result.

* **Forward Error**: Measures difference between computed and exact solution:

$$
\text{Forward Error} = \|\hat{y} - y\| \quad \text{or} \quad \frac{\|\hat{y} - y\|}{\|y\|}
$$

* **Backward Error**: Finds the smallest input perturbation $\Delta x$ such that the computed solution is exact for perturbed input:

$$
\hat{y} = f(x + \Delta x)
$$

An algorithm is **backward stable** if the relative backward error $\frac{\|\Delta x\|}{\|x\|} = \mathcal{O}(\epsilon_{\text{mach}})$.
Backward stability guarantees that errors in the output stem solely from problem conditioning, not solver algorithmic instability.

### Backward Stability Example: Triangular Solves

A fundamental result by J.H. Wilkinson demonstrates that solving a triangular system $Tx = b$
via back-substitution in floating-point arithmetic is unconditionally backward stable.
The computed solution $\hat{x}$ exactly satisfies a perturbed system:

$$
(T + \delta T)\hat{x} = b
$$

where the perturbation matrix $\delta T$ is bounded component-wise by:

$$
|\delta T_{ij}| \le \frac{nu}{1 - nu} |T_{ij}|
$$

with $u = \epsilon_{\text{mach}}/2$ being the unit roundoff and $n$ the matrix dimension.
Consequently, in any matrix norm, $\|\delta T\| \le c_n u \|T\|$ for a small constant $c_n \approx n$.
This implies the algorithm introduces numerical errors no larger than what would arise
from storing the matrix $T$ in finite precision.

## 10. Condition Number & Error Amplification Proof

Let $A \in \mathbb{R}^{n \times n}$ be non-singular, $b \neq 0$, and consider the linear system $Ax = b$.
Suppose $b$ is perturbed by $\Delta b$, inducing an error $\Delta x$ in $x$, so $A(x + \Delta x) = b + \Delta b$.

### Step-by-Step Proof:

1. By linearity, $Ax + A \Delta x = b + \Delta b$.
   
   Since $Ax = b$, we obtain:

$$
A \Delta x = \Delta b \implies \Delta x = A^{-1} \Delta b
$$

2. Taking any consistent matrix norm on both sides:

$$
\|\Delta x\| = \|A^{-1} \Delta b\| \le \|A^{-1}\| \|\Delta b\|
$$

3. Similarly, for the unperturbed system $Ax = b$:

$$
\|b\| = \|Ax\| \le \|A\| \|x\| \implies \frac{1}{\|x\|} \le \frac{\|A\|}{\|b\|}
$$

4. Multiplying inequalities (2) and (3) yields:

$$
\frac{\|\Delta x\|}{\|x\|} \le \|A\| \|A^{-1}\| \frac{\|\Delta b\|}{\|b\|}
$$

Result:

$$
\boxed{\frac{\|\Delta x\|}{\|x\|} \le \|A\| \|A^{-1}\| \frac{\|\Delta b\|}{\|b\|} = \kappa(A) \frac{\|\Delta b\|}{\|b\|}}
$$

If both $A$ and $b$ are perturbed such that $(A + \Delta A)(x + \Delta x) = b + \Delta b$,
and assuming $\kappa(A) \frac{\|\Delta A\|}{\|A\|} < 1$, the general relative error bound is:

$$
\frac{\|\Delta x\|}{\|x\|} \le \frac{\kappa(A)}{1 - \kappa(A) \frac{\|\Delta A\|}{\|A\|}} \left( \frac{\|\Delta b\|}{\|b\|} + \frac{\|\Delta A\|}{\|A\|} \right)
$$

## 11. Thomas Algorithm for Tridiagonal Systems

For a tridiagonal system $Ax = b$ where $A = \operatorname{tridiag}(a_i, b_i, c_i)$,
Gaussian elimination without pivoting simplifies to the **Thomas Algorithm**:

1. Explicit initialization: $c_1' = \frac{c_1}{b_1}$ and $d_1' = \frac{d_1}{b_1}$.

2. Forward elimination: Modify coefficients $c_i' = \frac{c_i}{b_i - a_i c_{i-1}'}$ and $d_i' = \frac{d_i - a_i d_{i-1}'}{b_i - a_i c_{i-1}'}$ for $i = 2, \dots, n$.

3. Back substitution: $x_n = d_n'$, and $x_i = d_i' - c_i' x_{i+1}$ for $i = n-1, \dots, 1$.

Complexity is $\mathcal{O}(n)$ flops with $\mathcal{O}(n)$ memory storage.
The algorithm is unconditionally stable if $A$ is strictly diagonally dominant or SPD.

## 12. Governing Principles of Iterative Solvers

Direct solvers ($\mathcal{O}(n^3)$ flops, $\mathcal{O}(n^2)$ fill-in memory)
become infeasible for massive sparse systems.
Iterative solvers construct a sequence of approximations $\{x_k\}_{k=0}^\infty$
starting from an initial guess $x_0$ such that:

$$
\lim_{k \to \infty} x_k = x = A^{-1}b
$$

## 13. Stationary Iterative Methods

Stationary methods split $A = M - N$ with $M$ non-singular (easily invertible).
The system $Ax = b$ transforms into:

$$
(M - N) x = b \iff M x = N x + b \iff x = M^{-1}N x + M^{-1}b
$$

This induces the iterative scheme:

$$
M x_{k+1} = N x_k + b \implies x_{k+1} = M^{-1}N x_k + M^{-1}b
$$

Let $A = D - L - U$, where $D$ is the diagonal part of $A$,
$-L$ is the strictly lower triangular part, and $-U$ is the strictly upper triangular part.

### Classical Splits:

1. **Jacobi Method**: $M = D$, $N = L + U$.

$$
x_{k+1} = D^{-1}(L + U)x_k + D^{-1}b
$$

2. **Gauss-Seidel (GS) Method**: $M = D - L$, $N = U$.

$$
(D - L)x_{k+1} = U x_k + b \implies x_{k+1} = (D - L)^{-1} U x_k + (D - L)^{-1}b
$$

3. **Successive Over-Relaxation (SOR)**: Introduces a parameter $\omega \in (0, 2)$ to extrapolate Gauss-Seidel updates:

$$
M = \frac{1}{\omega}D - L, \quad N = \left(\frac{1}{\omega} - 1\right)D + U
$$

$$
\left(\frac{1}{\omega}D - L\right)x_{k+1} = \left[\left(\frac{1}{\omega} - 1\right)D + U\right] x_k + b
$$

### Convergence Theorems:

* **Diagonally Dominant Theorem**: If $A$ is strictly row diagonally dominant ($|a_{ii}| > \sum_{j \neq i} |a_{ij}|$),
  both Jacobi and Gauss-Seidel converge for any $x_0$.
* **Ostrowski-Reich Theorem**: If $A$ is SPD, SOR converges for any $x_0$ if and only if $\omega \in (0, 2)$.

## 14. Spectral Radius Convergence Criterion

Subtracting $M x_{k+1} = N x_k + b$ from $M x = N x + b$ yields:

$$
M(x - x_{k+1}) = N(x - x_k) \implies e_{k+1} = M^{-1}N e_k = G e_k
$$

where $G = M^{-1}N$ is the **iteration matrix**. By induction, $e_k = G^k e_0$.

### Proof of Convergence Criterion:

1. **Error Evolution**: From $e_{k+1} = G e_k$, applying the iteration recursively yields $e_k = G^k e_0$.

2. **Necessity and Sufficiency**: The error satisfies $e_k \to 0$ as $k \to \infty$
   for all initial errors $e_0 \in \mathbb{R}^n$ if and only if $\lim_{k \to \infty} G^k = 0$.

3. **Spectral Radius Link (Gelfand's Formula)**: By Gelfand's spectral radius formula, the asymptotic growth rate of matrix powers is given by:

$$
\lim_{k \to \infty} \|G^k\|^{\frac{1}{k}} = \rho(G)
$$

   where $\rho(G) = \max_i |\lambda_i(G)|$ is the spectral radius of $G$. Thus, $G^k \to 0$ as $k \to \infty$ if and only if $\rho(G) < 1$.

Result:

$$
\boxed{\lim_{k \to \infty} e_k = 0 \quad \forall e_0 \iff \rho(G) < 1}
$$

### Asymptotic Convergence Rate

While $\rho(G) < 1$ guarantees convergence, the speed is dictated by the **asymptotic rate of convergence**:

$$
R_\infty(G) = -\ln(\rho(G))
$$

The number of iterations required to reduce the initial error by a factor of $10^{-p}$
is approximately $p \ln(10) / R_\infty(G)$.

#### Model Problem Analysis (2D Poisson Equation)

Consider the 2D Poisson equation $-\Delta u = f$ discretized on a square domain with an $N \times N$ grid,
giving a system size $n = N^2$. The mesh spacing is $h = \frac{1}{N+1}$.

Spectral radii for the stationary methods take the form:

* **Jacobi**: $\rho(G_{\text{Jac}}) \approx 1 - \frac{\pi^2}{2} h^2 \implies R_\infty(G_{\text{Jac}}) \approx \frac{\pi^2}{2} h^2 = \mathcal{O}(h^2)$
* **Gauss-Seidel (GS)**: $\rho(G_{\text{GS}}) \approx 1 - \pi^2 h^2 \implies R_\infty(G_{\text{GS}}) \approx \pi^2 h^2 = \mathcal{O}(h^2)$
* **Optimal SOR**: $\rho(G_{\text{SOR}}) \approx 1 - 2\pi h \implies R_\infty(G_{\text{SOR}}) \approx 2\pi h = \mathcal{O}(h)$

GS converges twice as fast as Jacobi, but both require $\mathcal{O}(N^2)$ iterations.
Optimal SOR drastically reduces this to $\mathcal{O}(N)$ iterations, showcasing the power of parameter tuning.

## 15. Power Iteration for Eigenvalue Problems

To find the dominant eigenvalue $\lambda_1$ (where $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$)
and corresponding eigenvector $v_1$ of $A$:

1. Choose unit vector $q_0$ with $\|q_0\|_2 = 1$.

2. Iterate: $z_{k+1} = A q_k, \quad q_{k+1} = \frac{z_{k+1}}{\|z_{k+1}\|_2}$.

3. Approximate dominant eigenvalue via Rayleigh Quotient: $\mu_k = q_k^T A q_k \to \lambda_1$.

Convergence rate is linear: $\mathcal{O}\left(\left\vert\frac{\lambda_2}{\lambda_1}\right\vert^k\right)$.

## 16. Krylov Subspace Methods

Stationary methods rely only on local information from previous step $x_k$.
**Krylov subspace methods** construct approximations $x_k \in x_0 + \mathcal{K}_k(A, r_0)$
by extracting optimal vectors from the $k$-th Krylov subspace:

$$
\mathcal{K}_k(A, r_0) = \operatorname{span}\{r_0, A r_0, A^2 r_0, \dots, A^{k-1} r_0\}
$$

## 17. Conjugate Gradient (CG) & Step-by-Step Error Bound Proof

For a Symmetric Positive Definite (SPD) matrix $A \in \mathbb{R}^{n \times n}$,
CG minimizes the $A$-norm (energy norm) of the error $\|e_k\|_A = \sqrt{e_k^T A e_k}$ over $x_0 + \mathcal{K}_k(A, r_0)$.

### Step-by-Step Proof of the CG Error Bound:

1. **Polynomial Representation of Error**: Any $x_k \in x_0 + \mathcal{K}_k(A, r_0)$ can be written
   as $x_k = x_0 + q_{k-1}(A) r_0$ for some polynomial $q_{k-1}$ of degree at most $k-1$.
   
   Since $r_0 = b - A x_0 = A(x - x_0) = A e_0$:

$$
e_k = x - x_k = e_0 - q_{k-1}(A) A e_0 = (I - A q_{k-1}(A)) e_0 = p_k(A) e_0
$$

   where $p_k(t) = 1 - t q_{k-1}(t) \in \mathcal{P}_k$ satisfies $p_k(0) = 1$.

2. **Optimality Property**: CG finds the unique polynomial $p_k^{\ast} \in \mathcal{P}_k$ with $p_k^{\ast}(0) = 1$ that minimizes $\|p_k(A) e_0\|_A$:

$$
\|e_k\|_A = \min_{p_k \in \mathcal{P}_k, p_k(0)=1} \|p_k(A) e_0\|_A \le \left( \min_{p_k \in \mathcal{P}_k, p_k(0)=1} \|p_k(A)\|_A \right) \|e_0\|_A
$$

3. **Spectral Bound**: Since $A$ is SPD, it has real positive eigenvalues $0 < \lambda_{\min} \le \lambda_i \le \lambda_{\max}$.
   
   In the eigenbasis of $A$:

$$
\|p_k(A)\|_A = \max_{\lambda \in \sigma(A)} |p_k(\lambda)| \le \max_{\lambda \in [\lambda_{\min}, \lambda_{\max}]} |p_k(\lambda)|
$$

4. **Chebyshev Polynomial Choice**: To minimize the maximum value on $[\lambda_{\min}, \lambda_{\max}]$ while maintaining $p_k(0) = 1$, we choose the scaled and shifted Chebyshev polynomial of the first kind $T_k(x)$:

$$
p_k(\lambda) = \frac{T_k\left( \frac{\lambda_{\max} + \lambda_{\min} - 2\lambda}{\lambda_{\max} - \lambda_{\min}} \right)}{T_k\left( \frac{\lambda_{\max} + \lambda_{\min}}{\lambda_{\max} - \lambda_{\min}} \right)}
$$

   For $\lambda \in [\lambda_{\min}, \lambda_{\max}]$, the numerator argument lies in $[-1, 1]$, so $|T_k(\cdot)| \le 1$.
   
   Hence:

$$
\max_{\lambda \in [\lambda_{\min}, \lambda_{\max}]} |p_k(\lambda)| \le \frac{1}{T_k\left( \frac{\lambda_{\max} + \lambda_{\min}}{\lambda_{\max} - \lambda_{\min}} \right)} = \frac{1}{T_k\left( \frac{\kappa + 1}{\kappa - 1} \right)}
$$

   where $\kappa = \kappa_2(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$.

5. **Asymptotic Evaluation**: Using the explicit formula $T_k(w) = \frac{1}{2} \left[ (w + \sqrt{w^2 - 1})^k + (w - \sqrt{w^2 - 1})^k \right]$ for $w = \frac{\kappa + 1}{\kappa - 1}$:

$$
w + \sqrt{w^2 - 1} = \frac{\kappa + 1 + \sqrt{(\kappa + 1)^2 - (\kappa - 1)^2}}{\kappa - 1} = \frac{\kappa + 1 + 2\sqrt{\kappa}}{\kappa - 1} = \frac{(\sqrt{\kappa} + 1)^2}{(\sqrt{\kappa} + 1)(\sqrt{\kappa} - 1)} = \frac{\sqrt{\kappa} + 1}{\sqrt{\kappa} - 1}
$$

   Thus, $T_k(w) \ge \frac{1}{2} \left( \frac{\sqrt{\kappa} + 1}{\sqrt{\kappa} - 1} \right)^k$.
   
   Taking the reciprocal completes the proof.

Result:

$$
\boxed{\|e_k\|_A \le 2 \left( \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \right)^k \|e_0\|_A}
$$

## 18. GMRES for Non-symmetric Matrices

For general non-symmetric or indefinite matrices $A$, Generalized Minimal Residual (GMRES)
minimizes the 2-norm of the residual $\|r_k\|_2$ over $x_0 + \mathcal{K}_k(A, r_0)$.

### Mathematical Formulation via Arnoldi Iteration:

1. **Arnoldi Iteration**: Computes an orthonormal basis $V_k = [v_1, v_2, \dots, v_k] \in \mathbb{R}^{n \times k}$
   for $\mathcal{K}_k(A, r_0)$ and an upper Hessenberg matrix $\bar{H}_k \in \mathbb{R}^{(k+1) \times k}$ satisfying the fundamental relation:

$$
A V_k = V_{k+1} \bar{H}_k
$$

2. **Least Squares Transformation**: Express $x_k = x_0 + V_k y_k$ for $y_k \in \mathbb{R}^k$.
   
   The residual vector expands to:

$$
r_k = b - A(x_0 + V_k y_k) = r_0 - A V_k y_k = \|r_0\|_2 v_1 - V_{k+1} \bar{H}_k y_k = V_{k+1} (\beta e_1 - \bar{H}_k y_k)
$$

   where $\beta = \|r_0\|_2$ and $e_1 = [1, 0, \dots, 0]^T \in \mathbb{R}^{k+1}$.

3. **Orthonormal Preservation**: Since $V_{k+1}$ has orthonormal columns ($\|V_{k+1} z\|_2 = \|z\|_2$):

$$
\|r_k\|_2 = \min_{y_k \in \mathbb{R}^k} \|\beta e_1 - \bar{H}_k y_k\|_2
$$

This solves a small $(k+1) \times k$ linear least-squares problem using QR factorization
via Givens rotations in $\mathcal{O}(k^2)$ flops.
Because storing $V_k$ requires $\mathcal{O}(nk)$ memory, GMRES is restarted after $m$ steps, denoted as GMRES($m$).

### Proof of GMRES Polynomial Convergence Bound:

1. **Polynomial Representation of Residual**: Since $x_k \in x_0 + \mathcal{K}_k(A, r_0)$,
   any candidate solution can be expressed as $x_k = x_0 + q_{k-1}(A) r_0$
   for some polynomial $q_{k-1}$ of degree at most $k-1$.
   
   The corresponding residual is:

$$
r_k = b - A x_k = r_0 - A q_{k-1}(A) r_0 = (I - A q_{k-1}(A)) r_0 = p_k(A) r_0
$$

   where $p_k(t) = 1 - t q_{k-1}(t) \in \mathcal{P}_k$ satisfies $p_k(0) = 1$.

2. **Residual Minimization Optimality**: GMRES explicitly minimizes $\|r_k\|_2 = \|p_k(A) r_0\|_2$ over all $p_k \in \mathcal{P}_k$ with $p_k(0) = 1$:

$$
\|r_k\|_2 = \min_{p_k \in \mathcal{P}_k, p_k(0)=1} \|p_k(A) r_0\|_2 \le \left( \min_{p_k \in \mathcal{P}_k, p_k(0)=1} \|p_k(A)\|_2 \right) \|r_0\|_2
$$

3. **Diagonalizable Matrix Bound**: If $A$ is diagonalizable such that $A = V \Lambda V^{-1}$, then $p_k(A) = V p_k(\Lambda) V^{-1}$, yielding:

$$
\|p_k(A)\|_2 \le \|V\|_2 \|p_k(\Lambda)\|_2 \|V^{-1}\|_2 = \kappa_2(V) \max_{\lambda \in \sigma(A)} |p_k(\lambda)|
$$

Result:

$$
\boxed{\|r_k\|_2 \le \kappa_2(V) \left( \min_{p_k \in \mathcal{P}_k, p_k(0)=1} \max_{\lambda \in \sigma(A)} |p_k(\lambda)| \right) \|r_0\|_2}
$$

Convergence is rapid when the eigenvalues $\sigma(A)$ are tightly clustered away from the origin
and $A$ is not highly non-normal (i.e., $\kappa_2(V)$ is small).

## 19. Preconditioning

If $\kappa(A) \gg 1$, iterative methods converge very slowly. **Preconditioning** transforms $Ax = b$
into an equivalent system with superior spectral properties using a non-singular matrix $M \approx A$:

1. **Left Preconditioning**: $M^{-1}Ax = M^{-1}b$.

2. **Right Preconditioning**: $A M^{-1} y = b$, with $x = M^{-1}y$.

3. **Split (Symmetric) Preconditioning**: For SPD systems $A$ and SPD preconditioner $M = L L^T$,
   solve $L^{-1} A L^{-T} y = L^{-1} b$ with $x = L^{-T} y$.

### Key Requirements for $M$:

* $M^{-1}v$ must be computationally cheap to evaluate ($\mathcal{O}(n)$ or $\mathcal{O}(n \log n)$ flops).
* $\kappa(M^{-1}A) \ll \kappa(A)$, or eigenvalues of $M^{-1}A$ are tightly clustered around 1.

### Standard Preconditioners:

* **Jacobi (Diagonal)**: $M = \operatorname{diag}(A)$.
* **Incomplete LU (ILU)**: $M = \tilde{L}\tilde{U} \approx A$, dropping fill-in elements outside a sparsity pattern.
* **Incomplete Cholesky (IC)**: For SPD matrices, $M = \tilde{L}\tilde{L}^T$.
* **Algebraic Multigrid (AMG)**: Hierarchical coarse-grid corrections targeting low-frequency error components.

## 20. Multigrid Methods Theory

Classical stationary methods (like Jacobi and Gauss-Seidel) suffer from a phenomenon called **critical slowing down**.
They rapidly damp out oscillatory (high-frequency) error components but struggle immensely
with smooth (low-frequency) error components, taking $\mathcal{O}(n^2)$ iterations to converge for standard elliptic PDEs.

### The Multigrid Principle

Multigrid methods overcome this by combining two complementary processes:

1. **Smoothing (Relaxation)**: Applying a few steps of Jacobi or Gauss-Seidel to rapidly eliminate high-frequency errors on the fine grid.

2. **Coarse-Grid Correction**: What appears as a smooth, low-frequency error on a fine mesh
   becomes an oscillatory, high-frequency error on a coarser mesh.
   
   We solve the residual equation on a coarser grid to efficiently eliminate these low-frequency modes.

### Inter-Grid Operators

To move between hierarchical grids, we define:

* **Restriction Operator $R$**: Transfers vectors from the fine grid $\Omega_h$ to the coarse grid $\Omega_{2h}$ (e.g., full weighting).
* **Prolongation Operator $P$**: Interpolates vectors from the coarse grid $\Omega_{2h}$ back to the fine grid $\Omega_h$ (e.g., linear interpolation). Often, $R = c P^T$ for some scalar $c$.

### Two-Grid Cycle

The basic Two-Grid cycle for solving $A_h x_h = b_h$ is:

1. **Pre-smoothing**: Apply $\nu_1$ steps of a smoother (e.g., Gauss-Seidel) on $A_h x_h = b_h$ to get an approximation $\tilde{x}_h$.

2. **Residual computation**: $r_h = b_h - A_h \tilde{x}_h$.

3. **Restriction**: Project the residual to the coarse grid: $r_{2h} = R r_h$.

4. **Coarse-Grid Solve**: Solve the residual equation exactly on the coarse grid: $A_{2h} e_{2h} = r_{2h}$.

5. **Prolongation & Correction**: Interpolate the error back to the fine grid
   and correct the solution: $\hat{x}_h = \tilde{x}_h + P e_{2h}$.

6. **Post-smoothing**: Apply $\nu_2$ steps of a smoother to eliminate any high-frequency errors
   introduced by the prolongation step.

### V-Cycle and W-Cycle

Since solving the coarse grid exactly in step 4 is still expensive, the idea is applied recursively.
A coarse-grid solve calls the multigrid algorithm on an even coarser grid, down to a base level where a direct solver is trivial.

* **V-cycle**: Recursively visits coarser grids once.
* **W-cycle**: Recursively visits coarser grids twice per level, providing more robust coarse-grid correction.

### Complexity and Mesh-Independent Convergence

Because the number of variables shrinks geometrically (e.g., by a factor of 4 in 2D) on coarser grids,
the work per cycle is bounded by a geometric series.

Result:

$$
\boxed{\text{Complexity of one V-cycle is } \mathcal{O}(n)}
$$

More importantly, the convergence rate $\rho$ of a proper multigrid cycle is bounded strictly below 1, independent of the mesh size $h$.
This **mesh-independent convergence** makes multigrid an optimal $\mathcal{O}(n)$ solver for elliptic PDEs.

## 21. Verification Strategy

To verify the correct implementation and convergence of iterative solvers:

* **Relative Residual Check**: Use $\|r_k\|_2 / \|b\|_2 \le \operatorname{tol}$ as a robust stopping criterion,
  rather than just the error between successive iterates.
* **Direct Method Baseline**: Compare the iterative solution against a small dense system
  solved directly (e.g., using LU factorization) to ensure exact correctness.
* **Convergence Rate Validation**: Track the convergence rate and verify it matches theoretical predictions
  (e.g., spectral radius $\rho(G)$ for stationary methods).
* **Condition Number Monitoring**: Monitor $\kappa(A)$, which indicates the expected difficulty and convergence speed of the problem.
* **Symmetry and Definiteness**: Verify prerequisites before applying specific solvers
  (e.g., ensuring the matrix is Symmetric Positive-Definite for Conjugate Gradient).
* **Cross-Validation with Initial Guesses**: Test the solver with different initial guesses $x_0$
  to confirm it converges stably to the identical unique solution.

## 22. Real-World Applications

* **Elliptic Partial Differential Equations**: Finite Element Method (FEM) and Finite Difference Method (FDM)
  discretizations yield sparse SPD stiffness matrices solved via PCG.
* **Google PageRank**: Dominant eigenvector calculation for link graph transition matrix
  $P \in \mathbb{R}^{10^{10} \times 10^{10}}$ using Power Iteration.
* **Computational Fluid Dynamics (CFD)**: Incompressible Navier-Stokes pressure Poisson equation
  solved via GMRES/BiCGSTAB.

## 23. AI & Machine Learning Connections

* **Gradient Descent as Stationary Iteration**: Quadratic optimization $f(x) = \frac{1}{2}x^T A x - b^T x$ has gradient $\nabla f(x) = Ax - b = -r(x)$.
  Gradient Descent $x_{k+1} = x_k - \alpha \nabla f(x_k) = (I - \alpha A) x_k + \alpha b$ is a stationary iteration with $M = \frac{1}{\alpha} I$.
* **Newton-CG and Hessian-Free Optimization**: In deep learning and large-scale optimization,
  computing exact Hessians $H = \nabla^2 \mathcal{L}(\theta)$ is impossible.
  Newton-CG uses CG to solve $H \Delta \theta = -\nabla \mathcal{L}(\theta)$ using matrix-free Hessian-vector products $H v$.
* **Preconditioning in Optimization**: K-FAC (Kronecker-factored Approximate Curvature) and
  adaptive optimizers (Adam, RMSprop) serve as preconditioners, scaling search directions
  by approximations of the Fisher information matrix / Hessian to tackle ill-conditioned loss landscapes.